# 데이터 분석 과제 · K-pop & Olist GitHub 분석과 차트 시각화

**과제:** ChatGPT / Claude 분석을 참고하여 GitHub 프로젝트 2개를 살피고, 새 분석기법을 제안하고, 저장한 그래프와 Streamlit 결과물을 구성한다.

**대상 저장소**
- K-pop: https://github.com/dominickim77-ui/mbc-kpop-data  
  ※ 2026-11-09 다시 확인할 때 404로 접근되지 않음. 아래의 K-pop 계산은 **사용자가 실제 CSV를 넣은 경우에만** 실행한다. 임의 데이터로 K-pop 통계를 만들어 내지 않는다.
- Olist: https://github.com/journal-izm/Olist-Data-Analysis  
  검증 가능한 원본 `rag/olist_facts.csv`의 F08, F14, F23~F25, F27 자료를 사용.

**차트 출처:** Olist 수업 검증 사실표. 새로운 모델 실험을 돌린 수치가 아니라 **기존 학습·검증 결과를 재시각화**한 것이다. 일부 지역 '그 외' 33.4%는 `100 - 42.0 - 12.8 - 11.8`로 계산했다. 

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from IPython.display import display, Image
from generate_charts import draw

ROOT = Path.cwd()
FACTS = ROOT / 'data' / 'olist_verified_facts.csv'
assert FACTS.exists(), '폴더 구조를 유지하고 노트북을 실행해 주세요.'
facts = pd.read_csv(FACTS)
print('검증된 사실표 행:',len(facts))
display(facts[['source_id','topic','group','value','unit']])

검증된 사실표 행: 14


   source_id        topic          group    value unit
0        F14       배송과 리뷰          제때 도착    4.300    점
1        F14       배송과 리뷰          늦게 도착    2.570    점
2        F23  감성분석 모델 정확도          전부 긍정    0.710   비율
3        F23  감성분석 모델 정확도  TF-IDF + 로지스틱    0.917   비율
4        F24  감성분석 모델 정확도  Bigram + 로지스틱    0.926   비율
5        F25  감성분석 모델 정확도   Embedding 평균    0.924   비율
6        F08    지역별 주문 비중             SP   42.000    %
7        F08    지역별 주문 비중             RJ   12.800    %
8        F08    지역별 주문 비중             MG   11.800    %
9        F08    지역별 주문 비중            그 외   33.400    %
10       F27       판매자 군집          핵심 우량  876.000    명
11       F27       판매자 군집          신규 성장  947.000    명
12       F27       판매자 군집             휴면  461.000    명
13       F27       판매자 군집         저평점 위험  218.000    명

## 분석 전 전처리와 신뢰성 확인

**질문:** 같은 단위, 같은 대상·시점을 비교하는가?  
Olist 사실표의 모든 숫자에는 사실ID를 연결한다. 병합이 필요한 경우 주문 `order_id`별 결제를 먼저 합산해 중복 매출을 방지한다. 원본 개별 주문 기록과 요약 사실표는 구분한다.

In [2]:
assert not facts[['source_id','topic','group']].duplicated().any(), '사실ID·항목 중복 확인 필요'
assert facts['value'].notna().all() and (facts['value']>=0).all()
regions=facts[facts.topic=='지역별 주문 비중']
print('지역별 주문 비중 합계:',round(regions.value.sum(),3),'%')
assert abs(regions.value.sum()-100)<0.01
print('데이터 품질 점검 통과')

지역별 주문 비중 합계: 100.0 %
데이터 품질 점검 통과


## 차트 ① 배송 지연과 리뷰 점수

**질문:** 늦게 도착한 주문의 평균 평점은 얼마나 낮은가?  
**한 줄 결과:** 제때 도착한 주문 4.30점, 늦게 도착한 주문 2.57점으로 **1.73점 낮았다**. 배송 지연이 점수를 떨어뜨린 원인이라고 인과관계까지 주장하지 않는다. [F14]

In [3]:
paths=draw()  # 4개의 과제용 PNG 생성
assert len(paths)>=3
print('저장한 차트 파일:',*[p.name for p in paths],sep='\n - ')
display(Image(filename=str(paths[0])))

저장한 차트 파일:
 - 01_배송과_리뷰.png
 - 02_감성분석_모델비교.png
 - 03_지역별_주문비중.png
 - 04_판매자_유형.png


<IPython.core.display.Image object>

## 차트 ② 감성분석 정확도 비교

**질문:** 딥러닝이 단순 머신러닝보다 반드시 정확한가?  
**한 줄 결과:** Bigram 머신러닝 **92.6%**, 배포용 Embedding 평균 **92.4%**로, 0.2%p 차이였다. 7,428건 테스트 기록의 정확도이며 본 노트북에서 모델을 재평가한 값은 아니다. [F23~F25]

In [4]:
display(Image(filename=str(paths[1])))
models=facts[facts.topic=='감성분석 모델 정확도'][['group','value']]
print(models.to_string(index=False))

<IPython.core.display.Image object>

        group  value
        전부 긍정  0.710
TF-IDF + 로지스틱  0.917
Bigram + 로지스틱  0.926
 Embedding 평균  0.924


## 차트 ③ 지역별 주문 비중

**질문:** 어느 지역에서 주문이 가장 많이 발생했는가?  
**한 줄 결과:** SP(42.0%), RJ(12.8%), MG(11.8%) 상위 3개 주의 합계는 **66.6%**다. 나머지 33.4%는 전체 100%에서 뺀 계산값이다. [F08]

In [5]:
display(Image(filename=str(paths[2])))

<IPython.core.display.Image object>

## 차트 ④ 판매자 군집 분석 (추가 차트)

**질문:** 판매자는 어떤 집단에 가장 많이 속하는가?  
**한 줄 결과:** 2,502명 중 **신규 성장 947명**이 가장 많았고 **저평점 위험 218명**이 가장 적었다. 군집 번호에 본래 뜻이 있는 것은 아니며 사람이 결과를 해석해 이름을 붙인다. [F27]

In [6]:
display(Image(filename=str(paths[3])))

<IPython.core.display.Image object>

## K-pop: Logarithmic Scale vs StandardScaler — 사용자가 실제 CSV를 넣은 경우에만

- **`ax.set_yscale('log')`**: 차트의 세로축 눈금만 로그 형태로 **표시**한다. 원본 값은 그대로다. 0/음수 데이터는 로그축에 직접 표시할 수 없다.
- **`np.log1p(x)`**: 실제 수치에 로그 변환을 적용한다. 0은 그대로 0이다. 왜도가 큰 조회수처럼 큰 값을 압축하는 데 쓴다.
- **`StandardScaler` (z-score)**: 값의 평균을 0, 표준편차를 1로 맞추어 **모델 입력 변수의 상대적 크기**를 비교하는 데 쓴다. 분포의 오른쪽 꼬리를 없애 주지는 않는다.
- Spotify 주간 스트리밍과 YouTube 누적 조회수는 집계 기간이 달라서 직접 크기를 비교하면 안 된다. **같은 플랫폼의 같은 기간 데이터**에서 추세·순위를 비교해야 한다.

아래 셀은 실측 K-pop CSV 파일을 직접 넣은 뒤 실행한다. 없으면 건너뛰며 **가상 차트를 실제 결과처럼 생성하지 않는다.**

In [7]:
kpop_path=ROOT/'kpop_comparison.csv'
if not kpop_path.exists():
    print('SKIP: 실제 K-pop CSV 없음. GitHub 저장소 접근 확인 후 kpop_comparison.csv를 같은 폴더에 넣어 주세요.')
else:
    df=pd.read_csv(kpop_path)
    print('실제 CSV 행 수:',len(df),'열 목록:',list(df.columns))
    display(df.head())
    # 문자열 숫자는 변환하지 않고 CSV의 숫자형 열만 선택한다.
    numeric=df.select_dtypes(include='number').columns.tolist()
    if not numeric:
        print('숫자형 열이 없습니다. 수집/전처리 단계의 열 타입을 먼저 확인하세요.')
    else:
        col=numeric[0]  # 확인 후 실제 분석 대상 숫자 열로 바꾸세요.
        x=df[col].dropna().astype(float)
        if (x<0).any() or len(x)<2:
            print('음수 또는 데이터 부족: 분석 보류')
        else:
            import matplotlib.pyplot as plt
            z=(x-x.mean())/x.std(ddof=0) if x.std(ddof=0)>0 else x*0
            fig,ax=plt.subplots(1,3,figsize=(14,4))
            for a,series,title in zip(ax,[x,np.log1p(x),z],['원본 값','log1p 변환','표준점수(z)']):
                a.hist(series,bins=min(12,len(series)));a.set_title(title);a.set_ylabel('데이터 수')
            fig.suptitle('K-pop 실측 CSV — Log 변환과 표준화 차이')
            fig.tight_layout()
            fig.savefig(ROOT/'charts'/'05_KPOP_실측_변환비교.png',dpi=170)
            plt.show()
            print('차트 저장: charts/05_KPOP_실측_변환비교.png')

SKIP: 실제 K-pop CSV 없음. GitHub 저장소 접근 확인 후 kpop_comparison.csv를 같은 폴더에 넣어 주세요.


## 두 저장소에 추가할 기법 (제안)

| 도메인 | 추가할 분석기법 | 필요한 데이터 / 적용 조건 | 검증 포인트 |
|---|---|---|---|
| K-pop | 로그축 대 원본축 비교 | 동일 플랫폼의 조회수·스트리밍, 0값 확인 | 로그축에서 과장되거나 왜곡된 인상 점검 |
| K-pop | Spearman 순위 상관 | 같은 집계기간, 관측 곡 ≥ 10 | 동시 변화일 뿐 인과가 아님 |
| K-pop | 주별 증가량·이동평균 | 영상별/곡별 여러 시점의 누적 조회수 | 증가량은 누적값을 차분, 음수/수정값 확인 |
| Olist | 배송 지연과 리뷰 비교 | 주문 기준 중복 제거 | 모집단·주문/리뷰 연결 확인 |
| Olist | Log1p + StandardScaler 비교 | 금액/주문수 등 오른쪽 꼬리 변수 | train에만 fit, test 누수 금지 |
| Olist | 임계값별 F1·재현율 | 리뷰 라벨·모델 확률·테스트 분리 | 0.5를 기본값으로 설정하고 오분류 검토 |
| Olist | 신규 고객·판매자 군집 | 반복 측정과 기간/대상 일치 | KMeans 군집명은 추후 해석 |

### Streamlit 모델 실행

`python download_models.py`로 **GitHub 원본 모델 파일**을 가져오고, 검증한 뒤 `streamlit run streamlit_app.py`를 실행한다. 모델은 훈련에 사용된 Tokenizer(단어 ID)와 `MAX_LEN=50`, `padding='post'`, `truncating='post'`를 그대로 사용해야 한다. 한국어 리뷰는 모델 훈련 언어가 아니어서 판단을 보류한다.

**한 줄 결론:** 숫자가 크다고 무조건 Log를 쓰지 않는다. 먼저 비교 질문, 측정 단위, 자료 분포, 모델 목적을 점검하고 그에 맞는 변환과 차트를 선택한다.

## 실제 공개자료 추가 실습 · Spotify 한국 주간 차트 (2026-11-05)

K-pop 저장소 `dominickim77-ui/mbc-kpop-data`는 재조회 시 404입니다. 원본 5곡 CSV와 **다른 별도 데이터셋**인 [Kworb 한국 Spotify 주간 TOP 20](https://www.kworb.net/spotify/country/kr_weekly.html)을 이용했습니다. 전체 국내 차트이므로 K-pop만으로 필터링된 차트가 **아닙니다**.

- 질문 1: 2026년 11월 5일 국내 주간 차트 상위 10곡 재생수는?
- 질문 2: 전주 대비 가장 많이 상승·하락한 곡은? (신규진입은 제외)
- 질문 3: 상위 20곡 내에서 상위 5곡 비중은?

**중요:** 전주 수치는 Kworb에 표시된 증감량으로 역산한 값이며, 확인된 19개 기존 곡에만 적용합니다. 원본 곡별 5곡 Spotify/YouTube 상호 비교와 혼용하지 않습니다.

In [8]:
import pandas as pd
from IPython.display import display, Image
from pathlib import Path
KPOP_CSV = Path('data/spotify_kr_weekly_2026-11-05_top20.csv')
kpop = pd.read_csv(KPOP_CSV)
assert len(kpop) == 20 and kpop['rank'].is_unique
assert kpop['weekly_streams'].gt(0).all()
print('자료:',kpop['chart_week_end'].iloc[0], '| 순위:',len(kpop),'곡 | 신규:',kpop['weekly_streams_change'].isna().sum())
display(kpop[['rank','artist','song','weekly_streams','weekly_streams_change']].head(10))

자료: 2026-11-05 | 순위: 20 곡 | 신규: 1


   rank   artist  ... weekly_streams  weekly_streams_change
0     1      A2O  ...        1078815              -341750.0
1     2    ILLIT  ...         876051               521593.0
2     3      IVE  ...         492796                62565.0
3     4      머무르  ...         479232                41326.0
4     5    Jimin  ...         477861               -17713.0
5     6  RESCENE  ...         465214                -7360.0
6     7   SOYEON  ...         390833               -20421.0
7     8   Redoor  ...         372019                -9988.0
8     9  RESCENE  ...         362195                    NaN
9    10   CORTIS  ...         359841               -15941.0

[10 rows x 5 columns]

### 차트 ⑤ 주간 재생수 TOP 10

**질문:** 상위 10개 곡의 재생 규모는?  
**한 줄 결과:** 1위 A2O `NEON`의 주간 재생수는 **1,078,815회**다. 같은 국내 차트의 같은 주를 비교한다.

In [9]:
display(Image(filename='charts/05_Spotify_한국주간_상위10.png'))

<IPython.core.display.Image object>

### 차트 ⑥ 전주 대비 증감률

**질문:** 어떤 곡의 주간 스트리밍이 가장 크게 변화했나?  
**한 줄 결과:** 기존 순위 곡 중 가장 큰 증가율은 **ILLIT You’re Under Arrest +147.15%**, 최대 감소율은 **Jung Kook Seven (w/ Latto) -37.89%**다. 신규 진입 곡 1개는 제외.

In [10]:
valid = kpop.dropna(subset=['weekly_streams_change']).copy()
valid['previous_weekly_streams_check'] = valid.weekly_streams-valid.weekly_streams_change
valid['wow_pct_check'] = valid.weekly_streams_change / valid.previous_weekly_streams_check * 100
assert (valid.previous_weekly_streams_check > 0).all()
print('비교 가능:',len(valid),'곡 / 신규 제외:',len(kpop)-len(valid))
display(valid[['rank','artist','song','wow_pct_check']].sort_values('wow_pct_check',ascending=False).head(5))
display(Image(filename='charts/06_Spotify_주간증감률.png'))

비교 가능: 19 곡 / 신규 제외: 1


    rank  ... wow_pct_check
1      2  ...    147.152272
11    12  ...     26.791755
2      3  ...     14.542188
3      4  ...      9.437185
13    14  ...      5.570185

[5 rows x 4 columns]

<IPython.core.display.Image object>

### 차트 ⑦ 상위 20곡 내 비중

**질문:** 상위 20곡 재생수 중 상위 5곡은 얼마나 차지하나?  
**한 줄 결과:** 상위 5곡이 상위 20곡 합계의 **40.63%**를 차지한다. 전체 시장 점유율이나 K-pop 점유율은 아니다.

In [11]:
total = kpop.weekly_streams.sum()
top5 = kpop.head(5).weekly_streams.sum()
print('20곡 총 재생수:',f'{total:,}',' / 상위 5곡:',f'{top5:,}',' / 내부 비중:',f'{top5/total:.2%}')
display(Image(filename='charts/07_Spotify_TOP20_집중도.png'))

20곡 총 재생수: 8,380,754  / 상위 5곡: 3,404,755  / 내부 비중: 40.63%


<IPython.core.display.Image object>